In [ ]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, InvalidVideoId, NoTranscriptFound, VideoUnavailable
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
embedding = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id= "deepseek-ai/DeepSeek-V4-Flash", task = 'text-generation'
)

model = ChatHuggingFace(llm=llm, temperature=0.3)

In [ ]:
video_id = "LPZh9BOjkQs"
ytt_api = YouTubeTranscriptApi()


try:
    transcript_list = ytt_api.fetch(video_id, languages=['en'])
    
    transcript = " ".join(chunk.text for chunk in transcript_list)
    
except(TranscriptsDisabled, InvalidVideoId, NoTranscriptFound, VideoUnavailable):
    print("Transcript not available or invalid video ID.")
    transcript = None
    
except AttributeError as e:
    print("Transcript API method not found:", e)
    

In [ ]:
# Text Splitting

splitter = RecursiveCharacterTextSplitter(chunk_size = 900, chunk_overlap = 250)

chunks = splitter.create_documents([transcript])



In [ ]:
chunks[10]

In [ ]:
# Embedding generation and storing in vector store

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding        
)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['c944bdcb-7057-4651-aebe-9c03184aab20'])

In [ ]:
# RETRIEVER

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4},
    search_type = 'similarity'
)

In [ ]:
retriever

In [ ]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant. Answer only from the provided transcript context.
    If the context is insufficiant, just say you don't know.
    {context}
    Question : {Question}
    """,
    input_variables=['context', 'Question']
)

In [ ]:
question = 'What is LLM'

retrieved_docs = retriever.invoke(question)

In [ ]:
print(type(retrieved_docs))

print(retrieved_docs)

In [ ]:
context_text = '\n\n'.join(doc.page_content for doc in retrieved_docs)

In [ ]:
context_text

In [ ]:
final_prompt = prompt.invoke({'context': context_text, 'Question': question})

In [ ]:
result = model.invoke(final_prompt)

print(result.content)

In [4]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [5]:
def format_docs(retrieved_docs):
    context_text = '\n\n'.join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
parallel_chain = RunnableParallel({
    
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
}
    
)

In [ ]:
parallel_chain.invoke('What is LLM')

In [8]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | model | parser

In [ ]:
main_chain.invoke('Can u summarize the video')